# Exploración del bucket S3

Carga credenciales desde `.env`, inventaria el bucket y permite previsualizar cualquier archivo.

Estructura observada:
- `data/`: tablas de dimensión (`customers`, `products`, `branches`, …) y tablas de hechos particionadas por `year=/month=/day=` (`transactions`, `digital_events`, `call_center_interactions`, …).
- `data_backup_20260831/`: copia de respaldo de `data/`.
- `marketing_campaigns.csv` también suelto en la raíz (además de en `data/`).

## Setup

In [ ]:
import os
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import s3fs
from dotenv import load_dotenv

ROOT = Path.cwd() if (Path.cwd() / ".env").exists() else Path.cwd().parent
load_dotenv(ROOT / ".env", override=True)

BUCKET = os.environ["S3_BUCKET_NAME"]
REGION = os.environ["AWS_REGION"]

s3 = boto3.client("s3", region_name=REGION)
fs = s3fs.S3FileSystem(client_kwargs={"region_name": REGION})

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
BUCKET

## Verificar acceso

In [ ]:
resp = s3.list_objects_v2(Bucket=BUCKET, MaxKeys=5)
print("OK. KeyCount de prueba:", resp["KeyCount"])

## Inventario de objetos

In [ ]:
rows = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET):
    for o in page.get("Contents", []):
        rows.append({"key": o["Key"], "size": o["Size"], "last_modified": o["LastModified"]})

inv = pd.DataFrame(rows)
inv = inv[~inv["key"].str.endswith("/")]  # descartar "carpetas" vacías
name = inv["key"].str.rsplit("/", n=1).str[-1]
inv["ext"] = name.str.extract(r"(\.[^.]+(?:\.gz|\.bz2|\.zip)?)$", expand=False).fillna("<none>").str.lower()
inv["top"] = inv["key"].str.split("/").str[0]
inv["depth"] = inv["key"].str.count("/")
inv["size_mb"] = inv["size"] / 1e6

print(f"{len(inv):,} objetos · {inv['size'].sum() / 1e9:.2f} GB")
inv.head(20)

In [ ]:
def human(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"

summary_top = (inv.groupby("top")
                  .agg(files=("key", "size"), bytes=("size", "sum"),
                       first=("last_modified", "min"), last=("last_modified", "max"))
                  .sort_values("bytes", ascending=False))
summary_top["size"] = summary_top["bytes"].map(human)
summary_top.drop(columns="bytes")

In [ ]:
summary_ext = inv.groupby("ext").agg(files=("key", "size"), bytes=("size", "sum")).sort_values("bytes", ascending=False)
summary_ext["size"] = summary_ext["bytes"].map(human)
summary_ext.drop(columns="bytes")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
summary_top["bytes"].div(1e6).head(20).plot.barh(ax=axes[0], title="MB por prefijo raíz")
inv["size_mb"].plot.hist(bins=50, log=True, ax=axes[1], title="Distribución de tamaños (MB)")
axes[0].invert_yaxis()
plt.tight_layout()

### Árbol de prefijos

In [ ]:
def tree(prefix="", max_depth=3, max_children=15):
    """Imprime la jerarquía de 'carpetas' con conteo de archivos y tamaño."""
    sub = inv[inv["key"].str.startswith(prefix)]
    rel = sub["key"].str[len(prefix):]

    def walk(df_rel, sizes, level, indent):
        if level >= max_depth:
            return
        parts = df_rel.str.split("/", n=1)
        head = parts.str[0]
        is_dir = parts.str.len() > 1
        groups = pd.DataFrame({"head": head, "dir": is_dir, "size": sizes.values, "rest": parts.str[1].values})
        dirs = groups[groups["dir"]].groupby("head")
        for i, (h, g) in enumerate(dirs):
            if i >= max_children:
                print(f"{indent}  … {dirs.ngroups - max_children} carpetas más")
                break
            print(f"{indent}{h}/  ({len(g):,} archivos, {human(g['size'].sum())})")
            walk(g["rest"].reset_index(drop=True), g["size"].reset_index(drop=True), level + 1, indent + "    ")
        files = groups[~groups["dir"]]
        for _, r in files.head(max_children).iterrows():
            print(f"{indent}{r['head']}  ({human(r['size'])})")
        if len(files) > max_children:
            print(f"{indent}… {len(files) - max_children} archivos más")

    walk(rel.reset_index(drop=True), sub["size"].reset_index(drop=True), 0, "")

tree(max_depth=3)

## Helpers de lectura

In [ ]:
def read_any(key, nrows=1000, **kwargs):
    """Lee un objeto del bucket en un DataFrame según su extensión."""
    path = f"s3://{BUCKET}/{key}"
    k = key.lower()
    comp = "gzip" if k.endswith(".gz") else "bz2" if k.endswith(".bz2") else "zip" if k.endswith(".zip") else None
    base = k.removesuffix(".gz").removesuffix(".bz2").removesuffix(".zip")
    opts = {"storage_options": {"client_kwargs": {"region_name": REGION}}}

    if base.endswith(".parquet") or base.endswith(".pq"):
        df = pd.read_parquet(path, **opts, **kwargs)
        return df.head(nrows) if nrows else df
    if base.endswith(".csv") or base.endswith(".tsv") or base.endswith(".txt"):
        sep = "\t" if base.endswith(".tsv") else kwargs.pop("sep", None)
        return pd.read_csv(path, sep=sep, engine="python" if sep is None else "c",
                           nrows=nrows, compression=comp, encoding=kwargs.pop("encoding", "utf-8-sig"), **opts, **kwargs)
    if base.endswith(".jsonl") or base.endswith(".ndjson"):
        return pd.read_json(path, lines=True, nrows=nrows, compression=comp, **opts, **kwargs)
    if base.endswith(".json"):
        try:
            return pd.read_json(path, lines=True, nrows=nrows, compression=comp, **opts, **kwargs)
        except ValueError:
            df = pd.json_normalize(pd.read_json(path, compression=comp, typ="series", **opts).tolist())
            return df.head(nrows) if nrows else df
    raise ValueError(f"Extensión no soportada: {key}. Usá peek(key) para ver los bytes crudos.")


def peek(key, nbytes=2000):
    """Muestra los primeros bytes de un objeto (útil para formatos desconocidos)."""
    body = s3.get_object(Bucket=BUCKET, Key=key, Range=f"bytes=0-{nbytes - 1}")["Body"].read()
    try:
        print(body.decode("utf-8"))
    except UnicodeDecodeError:
        print(body[:200])


def profile(df):
    """Resumen de columnas: tipo, nulos, cardinalidad y un ejemplo."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(dropna=True),
        "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    })

## Catálogo de tablas en `data/`

Cada tabla es un CSV único (dimensión) o una carpeta particionada por fecha (hechos).

In [ ]:
DATA = "data/"
d = inv[inv["key"].str.startswith(DATA)].copy()
d["table"] = d["key"].str[len(DATA):].str.split("/").str[0].str.removesuffix(".csv")
dates = d["key"].str.extract(r"year=(\d{4})/month=(\d{2})/day=(\d{2})")
d["date"] = pd.to_datetime(dates[0] + "-" + dates[1] + "-" + dates[2], errors="coerce")

catalog = (d.groupby("table")
             .agg(files=("key", "size"), bytes=("size", "sum"),
                  partitioned=("date", lambda s: s.notna().any()),
                  date_min=("date", "min"), date_max=("date", "max"))
             .sort_values("bytes", ascending=False))
catalog["size"] = catalog["bytes"].map(human)
catalog.drop(columns="bytes")

In [ ]:
# Días faltantes por tabla particionada
for t, row in catalog[catalog["partitioned"]].iterrows():
    have = set(d.loc[d["table"] == t, "date"].dropna())
    missing = pd.date_range(row["date_min"], row["date_max"]).difference(pd.DatetimeIndex(sorted(have)))
    print(f"{t:28s} {len(have):5d} días · faltan {len(missing)}" + (f" (ej: {list(missing[:5].date)})" if len(missing) else ""))

In [ ]:
def table_keys(table, start=None, end=None):
    """Keys de una tabla; si es particionada, filtra por rango de fechas [start, end]."""
    sub = d[d["table"] == table]
    if sub["date"].notna().any():
        if start:
            sub = sub[sub["date"] >= pd.Timestamp(start)]
        if end:
            sub = sub[sub["date"] <= pd.Timestamp(end)]
    return sub.sort_values("key")["key"].tolist()


def read_table(table, start=None, end=None, nrows=None, **kwargs):
    """Lee una tabla completa (o un rango de fechas si es particionada) y concatena."""
    keys = table_keys(table, start, end)
    if not keys:
        raise ValueError(f"No hay archivos para {table!r} en ese rango")
    frames = []
    for k in keys:
        df_k = read_any(k, nrows=nrows, **kwargs)
        m = pd.Series(k).str.extract(r"year=(\d{4})/month=(\d{2})/day=(\d{2})").iloc[0]
        if m.notna().all():
            df_k["partition_date"] = pd.Timestamp(f"{m[0]}-{m[1]}-{m[2]}")
        frames.append(df_k)
    return pd.concat(frames, ignore_index=True)

In [ ]:
# Esquema de cada tabla (muestra de 200 filas de un archivo por tabla)
schemas = {}
for t in catalog.index:
    sample = read_any(table_keys(t)[0], nrows=200)
    schemas[t] = sample
    print(f"\n## {t}  ({sample.shape[1]} columnas)")
    print(", ".join(sample.columns))

In [ ]:
# Ejemplo: una semana de transacciones
tx = read_table("transactions", start="2025-01-01", end="2025-01-07")
print(tx.shape)
tx.head()

## Previsualizar un archivo

In [ ]:
# Archivos más grandes: suelen ser los datasets principales
inv.sort_values("size", ascending=False)[["key", "size_mb", "last_modified"]].head(20)

In [ ]:
# Elegí un archivo (por defecto, el más grande con un formato tabular conocido)
# Ej: KEY = "data/customers.csv"
tabular = inv[inv["ext"].str.contains(r"\.(?:csv|tsv|parquet|pq|jsonl?|ndjson|txt)", regex=True)]
KEY = tabular.sort_values("size", ascending=False)["key"].iloc[0] if len(tabular) else inv["key"].iloc[0]
KEY

In [ ]:
df = read_any(KEY, nrows=10_000)
print(df.shape)
df.head()

In [ ]:
profile(df)

In [ ]:
df.describe(include="all").T

## Descargar a local (opcional)

Guarda en `data/`, que está en `.gitignore`.

In [ ]:
def download(key, dest=ROOT / "data"):
    target = Path(dest) / key
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        s3.download_file(BUCKET, key, str(target))
    return target

# download(KEY)